Xây dựng công thức - Top Down single linkage

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.cluster.hierarchy import linkage, dendrogram

df = pd.read_csv("data_sample.csv")
X = df.values
n = len(X)
dist_matrix = np.zeros((n, n))
for i in range(n):
    for j in range(i+1, n):
        d = np.sqrt(np.sum((X[i] - X[j]) ** 2))
        dist_matrix[i, j] = d
        dist_matrix[j, i] = d

def silhouette_score_custom(X, labels):
    n = len(X)
    unique_labels = np.unique(labels)
    if len(unique_labels) <= 1:
        return -1
    scores = []
    for i in range(n):
        same = labels == labels[i]
        same_idx = np.where(same)[0]
        same_idx = same_idx[same_idx != i]

        if len(same_idx) > 0:
            a = np.mean([np.linalg.norm(X[i] - X[j]) for j in same_idx])
        else:
            a = 0
        b = np.inf
        for lab in unique_labels:
            if lab == labels[i]:
                continue
            other = np.where(labels == lab)[0]
            if len(other) > 0:
                dist = np.mean([np.linalg.norm(X[i] - X[j]) for j in other])
                b = min(b, dist)
        s = (b - a) / max(a, b) if max(a, b) > 0 else 0
        scores.append(s)
    return np.mean(scores)

def clusters_to_labels(clusters, n):
    labels = np.zeros(n, dtype=int)
    for i, c in enumerate(clusters):
        for p in c:
            labels[p] = i
    return labels

def cluster_diameter(cluster):
    max_d = 0
    for i in cluster:
        for j in cluster:
            max_d = max(max_d, dist_matrix[i, j])
    return max_d

def split_cluster(cluster):
    max_d = -1
    seed_a, seed_b = None, None
    for i in cluster:
        for j in cluster:
            if dist_matrix[i, j] > max_d:
                max_d = dist_matrix[i, j]
                seed_a, seed_b = i, j
    c1 = [seed_a]
    c2 = [seed_b]
    for p in cluster:
        if p in (seed_a, seed_b):
            continue
        if dist_matrix[p, seed_a] < dist_matrix[p, seed_b]:
            c1.append(p)
        else:
            c2.append(p)
    return c1, c2, max_d
clusters = [list(range(n))]
linkage_matrix = []
current_cluster_id = n
cluster_ids = {0: current_cluster_id}

best_k = -1
best_score = -1
best_labels = None
k_silhouette_scores = {}
while len(clusters) < n:
    diameters = [cluster_diameter(c) for c in clusters]
    idx = np.argmax(diameters)
    cluster_to_split = clusters[idx]
    if len(cluster_to_split) <= 1:
        break
    c1, c2, dist = split_cluster(cluster_to_split)
    id_parent = cluster_ids[idx]
    id_c1 = current_cluster_id + 1
    id_c2 = current_cluster_id + 2
    current_cluster_id += 2
    linkage_matrix.append([
        id_parent,
        id_parent,
        dist,
        len(cluster_to_split)
    ])

    clusters[idx] = c1
    clusters.append(c2)
    cluster_ids[idx] = id_c1
    cluster_ids[len(clusters) - 1] = id_c2

    k = len(clusters)
    if 2 <= k <= 10:
        labels = clusters_to_labels(clusters, n)
        score = silhouette_score_custom(X, labels)
        k_silhouette_scores[k] = score
        if score > best_score:
            best_score = score
            best_k = k
            best_labels = labels.copy()

print("Hierarchical Clustering - TOP DOWN (Single Linkage)")
print("\nSilhouette scores:")
print("-" * 40)
for k in sorted(k_silhouette_scores):
    s = k_silhouette_scores[k]
    if k == best_k:
        print(f"k={k:<3}  score={s:.4f}  *** BEST ***")
    else:
        print(f"k={k:<3}  score={s:.4f}")

print("-" * 40)
print(f"\nBest k = {best_k}")
print(f"Best silhouette = {best_score:.4f}")


Hierarchical Clustering - TOP DOWN (Single Linkage)

Silhouette scores:
----------------------------------------
k=2    score=0.8108  *** BEST ***
k=3    score=0.5574
k=4    score=0.3098
k=5    score=0.3255
k=6    score=0.3256
k=7    score=0.3257
k=8    score=0.3199
k=9    score=0.3090
k=10   score=0.3064
----------------------------------------

Best k = 2
Best silhouette = 0.8108


Xây dựng công thức - Top Down complete linkage

In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv("data_sample.csv")
X = df.values
n = len(X)

dist_matrix = np.zeros((n, n))
for i in range(n):
    for j in range(i + 1, n):
        d = np.linalg.norm(X[i] - X[j])
        dist_matrix[i, j] = d
        dist_matrix[j, i] = d

def silhouette_score_custom(X, labels):
    n = len(X)
    unique_labels = np.unique(labels)
    if len(unique_labels) <= 1:
        return -1
    scores = []
    for i in range(n):
        same = np.where(labels == labels[i])[0]
        same = same[same != i]
        a = np.mean([np.linalg.norm(X[i] - X[j]) for j in same]) if len(same) > 0 else 0
        b = np.inf
        for lab in unique_labels:
            if lab == labels[i]:
                continue
            other = np.where(labels == lab)[0]
            if len(other) > 0:
                dist = np.mean([np.linalg.norm(X[i] - X[j]) for j in other])
                b = min(b, dist)
        s = (b - a) / max(a, b) if max(a, b) > 0 else 0
        scores.append(s)
    return np.mean(scores)

def clusters_to_labels(clusters, n):
    labels = np.zeros(n, dtype=int)
    for i, c in enumerate(clusters):
        for p in c:
            labels[p] = i
    return labels

def cluster_diameter_complete(cluster):
    max_d = 0
    for i in cluster:
        for j in cluster:
            max_d = max(max_d, dist_matrix[i, j])
    return max_d

def split_cluster_complete(cluster):
    max_d = -1
    seed_a, seed_b = None, None
    for i in cluster:
        for j in cluster:
            if dist_matrix[i, j] > max_d:
                max_d = dist_matrix[i, j]
                seed_a, seed_b = i, j
    c1 = [seed_a]
    c2 = [seed_b]
    for p in cluster:
        if p in (seed_a, seed_b):
            continue
        d1 = max(dist_matrix[p, q] for q in c1)
        d2 = max(dist_matrix[p, q] for q in c2)
        if d1 < d2:
            c1.append(p)
        else:
            c2.append(p)
    return c1, c2, max_d
clusters = [list(range(n))]
best_k = -1
best_score = -1
best_labels = None
k_silhouette_scores = {}
while len(clusters) < n:
    diameters = [cluster_diameter_complete(c) for c in clusters]
    idx = np.argmax(diameters)
    cluster_to_split = clusters[idx]
    if len(cluster_to_split) <= 1:
        break
    c1, c2, dist = split_cluster_complete(cluster_to_split)
    clusters[idx] = c1
    clusters.append(c2)
    k = len(clusters)
    if 2 <= k <= 10:
        labels = clusters_to_labels(clusters, n)
        score = silhouette_score_custom(X, labels)
        k_silhouette_scores[k] = score
        if score > best_score:
            best_score = score
            best_k = k
            best_labels = labels.copy()

print("Hierarchical Clustering - TOP DOWN ")
print("\nSilhouette scores:")
print("-" * 40)
for k in sorted(k_silhouette_scores):
    s = k_silhouette_scores[k]
    if k == best_k:
        print(f"k={k:<3}  score={s:.4f}  *** BEST ***")
    else:
        print(f"k={k:<3}  score={s:.4f}")

print("-" * 40)
print(f"\nBest k = {best_k}")
print(f"Best silhouette = {best_score:.4f}")


Hierarchical Clustering - TOP DOWN 

Silhouette scores:
----------------------------------------
k=2    score=0.8108  *** BEST ***
k=3    score=0.5546
k=4    score=0.3005
k=5    score=0.3039
k=6    score=0.2755
k=7    score=0.2843
k=8    score=0.2689
k=9    score=0.2394
k=10   score=0.2294
----------------------------------------

Best k = 2
Best silhouette = 0.8108


Sử dụng thư viện - Top Down

In [3]:
from sklearn.cluster import BisectingKMeans
from sklearn.metrics import silhouette_score

best_k = -1
best_score = -1
scores = {}

for k in range(2, 11):
    model = BisectingKMeans(
        n_clusters=k,
        random_state=42
    )
    labels = model.fit_predict(X)
    score = silhouette_score(X, labels)

    scores[k] = score
    if score > best_score:
        best_k = k
        best_score = score

print("\nTOP-DOWN (Bisecting K-Means)")
for k, s in scores.items():
    if k == best_k:
        print(f"k={k}, silhouette={s:.4f}  *** BEST ***")
    else:
        print(f"k={k}, silhouette={s:.4f}")

print(f"\nBest k = {best_k}, best silhouette = {best_score:.4f}")



TOP-DOWN (Bisecting K-Means)
k=2, silhouette=0.8108  *** BEST ***
k=3, silhouette=0.5629
k=4, silhouette=0.3215
k=5, silhouette=0.3184
k=6, silhouette=0.3275
k=7, silhouette=0.3366
k=8, silhouette=0.3264
k=9, silhouette=0.3113
k=10, silhouette=0.3132

Best k = 2, best silhouette = 0.8108
